# NTNU Workshop — Kelvin lattice: FEM → X-ray → neural_xray

Self-paced notebook (run **after** the 45 min demo). Pipeline:

1. **`fem_lattice_simulator`** — indent a Kelvin lattice, export deformed geometry per timestep  
2. **`xray_projection_render`** (`scripts/render_projections.sh`) → `data/kelvin/renders/`  
3. **Stage dataset** — build `transforms_00.json`, `transforms_XX.json`, `transforms_00_to_XX.json`  
4. **`neural_xray`** — train canonical volumes + velocity field

**Runtime:** T4 GPU. Total time is typically **1–3+ hours** (FEM + render + training).

Repo: [NTNU_metamaterials_workshop](https://github.com/igrega348/NTNU_metamaterials_workshop)

## Common errors

**`KeyError: 'optimizers'`** — delete partial velocity-field checkpoints and re-run training:

```bash
rm -rf /content/NTNU_metamaterials_workshop/outputs/kelvin/xray_vfield/
```

In [ ]:
# @title 1. Install dependencies

%cd /content/
!pip install -q --upgrade pip
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

!curl -sL "https://github.com/igrega348/tiny-cuda-nn-wheels/releases/download/1.7.3/tinycudann-2.0.post75260124-cp312-cp312-linux_x86_64.whl" -o tinycudann.whl
!pip install -q tinycudann.whl --force-reinstall

!apt-get -qq install -y golang-go

import os
REPO = "/content/NTNU_metamaterials_workshop"
if os.path.isdir(REPO):
    !rm -rf {REPO}
!git clone --recurse-submodules -q https://github.com/igrega348/NTNU_metamaterials_workshop.git

%cd {REPO}/fem_lattice_simulator
!pip install -q -e .

%cd {REPO}/neural_xray/nerfstudio
!pip install -q -e .
%cd {REPO}/neural_xray/nerfstudio-xray/nerf-xray
!pip install -q -e .
%cd {REPO}/neural_xray/xray_projection_render
!pip install -q -e .

%load_ext tensorboard
print("Ready:", REPO)

In [ ]:
# @title 2. FEM — simulate Kelvin lattice indentation

import os
REPO = "/content/NTNU_metamaterials_workshop"
RUN_NAME = "workshop_colab"
FEM = f"{REPO}/fem_lattice_simulator"
RUN_DIR = f"{FEM}/runs/{RUN_NAME}"
os.makedirs(f"{RUN_DIR}/model", exist_ok=True)
os.makedirs(f"{RUN_DIR}/timesteps", exist_ok=True)
os.makedirs(f"{RUN_DIR}/yaml", exist_ok=True)

%cd {FEM}

# Smaller mesh / fewer steps than desktop run_pipeline.sh (Colab-friendly)
!python scripts/generate_lattice_from_yaml.py \
  --yaml lattice.yaml --out {RUN_DIR}/model/out.json \
  --nx 4 --ny 4 --nz 4 --subdivide 4

!python scripts/apply_indent_boundary_conditions.py \
  --in {RUN_DIR}/model/out.json --out {RUN_DIR}/model/out.json \
  --patch-cells-x 2 --patch-cells-y 2 --patch-placement center \
  --indent-uz -0.8 --indenter-uxuy-zero

!python -m src.main {RUN_DIR}/model/out.json \
  --ramp-steps 10 --output-prefix {RUN_DIR}/timesteps/{RUN_NAME} --output-every 2

!python scripts/json_to_yaml.py "{RUN_DIR}/timesteps/{RUN_NAME}_t*.json" \
  --radius-from-area --outdir {RUN_DIR}/yaml --overwrite

print("FEM YAML timesteps in", f"{RUN_DIR}/yaml")

In [ ]:
# @title 3. Render X-ray projections (xray_projection_render)

import glob, os, shutil
REPO = "/content/NTNU_metamaterials_workshop"
RUN_NAME = "workshop_colab"
DATA = f"{REPO}/data/kelvin"
YAML_DST = f"{DATA}/yaml"
YAML_SRC = f"{REPO}/fem_lattice_simulator/runs/{RUN_NAME}/yaml"
os.makedirs(YAML_DST, exist_ok=True)

for f in glob.glob(f"{YAML_SRC}/*.yaml"):
    shutil.copy2(f, YAML_DST)
print("Copied", len(glob.glob(f"{YAML_DST}/*.yaml")), "YAML files to", YAML_DST)

os.environ["YAML_GLOB"] = f"{RUN_NAME}_t*.yaml"
os.environ["VOLUME_RES"] = "128"
os.environ["RESOLUTION"] = "256"
!bash {REPO}/scripts/render_projections.sh

print("Renders:", glob.glob(f"{DATA}/renders/*/transforms.json"))

In [ ]:
# @title 4. Combine projections + transforms for neural_xray

REPO = "/content/NTNU_metamaterials_workshop"
RUN_NAME = "workshop_colab"
DATA = f"{REPO}/data/kelvin"

!python {REPO}/scripts/stage_kelvin_for_nerf.py \
  --renders-dir {DATA}/renders \
  --yaml-dir {DATA}/yaml \
  --out-dir {DATA}

In [ ]:
# @title 5. Train neural_xray (~30+ min on T4)

REPO = "/content/NTNU_metamaterials_workshop"
!bash {REPO}/scripts/train_kelvin_workshop.sh

In [ ]:
# @title 6. TensorBoard
%tensorboard --logdir /content/NTNU_metamaterials_workshop/outputs/kelvin/
